In [1]:
!pip install -U langchain langchain-community langchain-openai langchain-chroma chromadb pypdf openai sentence-transformers


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importações básicas
import os
import shutil
import requests

# loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Banco vetorial (use apenas este)
from langchain_chroma import Chroma

# LLM (LM Studio / OpenAI-compatible)
from langchain_openai import ChatOpenAI

# Embeddings locais (sem OpenAI / sem LM Studio)
from sentence_transformers import SentenceTransformer

c:\Users\Administrador\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pdf_path = r"C:\Users\Administrador\OneDrive\Documentos\Profissional\Alura\2_Engenharia_IA\Arquitetura_RAG\PDFs\regras_futebol.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Páginas carregadas:", len(documents))
print("Exemplo:", documents[0].page_content[:200])

Páginas carregadas: 232
Exemplo: 1
Regras 
do Jogo
23/24Baixe o app
das Regras do Jogo


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print("Chunks:", len(chunks))
print("Exemplo chunk:", chunks[0].page_content[:200])
print("Metadata:", chunks[0].metadata)

Chunks: 375
Exemplo chunk: 1
Regras 
do Jogo
23/24Baixe o app
das Regras do Jogo
Metadata: {'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)', 'creator': 'Adobe InDesign 18.3 (Windows)', 'creationdate': '2023-06-29T16:28:35+02:00', 'trapped': '/False', 'moddate': '2023-07-05T19:18:02+00:00', 'source': 'C:\\Users\\Administrador\\OneDrive\\Documentos\\Profissional\\Alura\\2_Engenharia_IA\\Arquitetura_RAG\\PDFs\\regras_futebol.pdf', 'total_pages': 232, 'page': 0, 'page_label': '1'}


In [10]:
# Modelo de embeddings local (rápido e leve)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # 384 dims

texts = [d.page_content for d in chunks]
metadatas = [d.metadata for d in chunks]

# Gera embeddings localmente
vectors = embed_model.encode(texts, normalize_embeddings=True).tolist()

# Cria Chroma persistente e adiciona
vectorstore = Chroma(
    collection_name="regras_futebol",
    persist_directory=persist_dir
)

vectorstore._collection.add(
    ids=[str(i) for i in range(len(texts))],
    documents=texts,
    metadatas=metadatas,
    embeddings=vectors
)

print("Indexação concluída. Total docs:", len(texts))

Indexação concluída. Total docs: 375


In [11]:
pergunta = "Um jogador pode usar a mão para marcar um gol?"

qvec = embed_model.encode(pergunta, normalize_embeddings=True).tolist()

res = vectorstore._collection.query(
    query_embeddings=[qvec],
    n_results=4,
    include=["documents", "metadatas", "distances"]
)

print("Top trechos recuperados:", len(res["documents"][0]))
print("Distâncias:", res["distances"][0])

Top trechos recuperados: 4
Distâncias: [0.8578066229820251, 0.9087119102478027, 0.9465312957763672, 0.9664162993431091]


In [12]:
print("\nTrechos usados:\n")
for i, (doc, meta, dist) in enumerate(zip(res["documents"][0], res["metadatas"][0], res["distances"][0]), start=1):
    print(f"--- Trecho {i} ---")
    print("Distância:", dist)
    print("Fonte:", meta.get("source", "N/A"))
    print("Página:", meta.get("page", "N/A"))
    print(doc[:400])
    print()


Trechos usados:

--- Trecho 1 ---
Distância: 0.8578066229820251
Fonte: C:\Users\Administrador\OneDrive\Documentos\Profissional\Alura\2_Engenharia_IA\Arquitetura_RAG\PDFs\regras_futebol.pdf
Página: 106
não ser que haja uma clara oportunidade de gol. O árbitro deverá expulsar o 
jogador quando a bola estiver fora de jogo. Porém, se o jogador tocar na bola, 
disputá-la com um adversário ou interferir nele, o árbitro paralisará o jogo, 
expulsará o jogador e reiniciará o jogo com um tiro livre indireto, a não ser que o 
jogador tenha cometido uma infração mais grave.
Regras do Jogo 2023/24  |  Regr

--- Trecho 2 ---
Distância: 0.9087119102478027
Fonte: C:\Users\Administrador\OneDrive\Documentos\Profissional\Alura\2_Engenharia_IA\Arquitetura_RAG\PDFs\regras_futebol.pdf
Página: 155
·
 um jogador, um substituto, um jogador substituído ou expulso, ou um 
membro da comissão técnica da equipe que marcou o gol. O jogo deve ser 
reiniciado com um tiro livre direto da posição em que se encontrava 